# Practical Work 3

## Loading the packages

In [ ]:
import numpy as np
import matplotlib.pyplot as pl ## hello

## The Dataset


In [ ]:
%pip install ipympl

In [ ]:
import pandas as pd

 # Change this to any folder where data is if needed
mouse1 = pd.read_csv('EEG_mouse_data_1.csv')
mouse2 = pd.read_csv('EEG_mouse_data_2.csv')
dataset = pd.concat([mouse1, mouse2])
test_data = pd.read_csv('EEG_mouse_data_test.csv')
dataset

<font color="red">**For it to work on Colab, you will need to reload your session (Exécution -> redémarrer la session)**</font>

<font color="orange">**Make sure to put a large amount of points otherwise the cross validation folds will be really small**</font>

## Show the dataset

In [ ]:
dataset = np.array(dataset)
test_data = np.array(test_data)
dataset

In [ ]:
input_data = dataset[:, 1:11].astype(float)
feature_mean = input_data.mean(axis=0)
feature_std = input_data.std(axis=0) + 1e-8
input_data = (input_data - feature_mean) / feature_std  # Standardize each feature

state_to_class = {"w": 0, "r": 1, "n": 2}
output_data = np.array([state_to_class[state] for state in dataset[:, 0]], dtype=int)

# test_data has only features (no state column in EEG_mouse_data_test.csv)
test_input_data = test_data[:, :10].astype(float)
test_input_data = (test_input_data - feature_mean) / feature_std

val_input = test_input_data
val_output = None

In [ ]:
input_data

In [ ]:
output_data

In [ ]:
# Check data balance
class_names = ["Awake", "REM", "Non-REM"]
unique, counts = np.unique(output_data, return_counts=True)
for label, count in zip(unique, counts):
    state = class_names[label]
    print(f"{state}: {count} samples ({count/len(output_data)*100:.1f}%)")

In [ ]:
%matplotlib inline

In [ ]:
import keras
from keras import layers

pl.clf()

keras.utils.set_random_seed(123)

class_colors = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}
validation_color = "lightgray"
class_names = ["Awake", "REM", "Non-REM"]

pl.figure(figsize=(4,4))

# Plot train data and validation data
pl.scatter(input_data[:,0], input_data[:,1], c=[class_colors[int(d)] for d in output_data], s=100)
pl.scatter(val_input[:,0], val_input[:,1], c=validation_color, s=100)
pl.title('Training data (colored) and validation data (gray).')
pl.show()

In [ ]:
MODEL_CONFIG = {
  "input_dim": 10,
  "hidden_units": [32],
  "learning_rate": 5e-4,
  "batch_size": 128,
  "epochs": 300,
  "early_stopping_patience": 15,
  "reduce_lr_patience": 5,
  "reduce_lr_factor": 0.5,
  "min_lr": 1e-5,
}

def create_model(config=MODEL_CONFIG):
  # Keep this config aligned with the folds notebook
  mlp = keras.Sequential()
  mlp.add(layers.Input(shape=(config["input_dim"],)))

  for units in config["hidden_units"]:
      mlp.add(layers.Dense(units, activation="relu"))

  mlp.add(layers.Dense(3, activation="softmax"))

  mlp.compile(
      optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
      loss="sparse_categorical_crossentropy",
      metrics=["accuracy"],
  )

  return mlp

mlp = create_model()
print("MODEL_CONFIG:", MODEL_CONFIG)
mlp.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

monitor_metric = "val_loss" if val_output is not None else "loss"

early_stopping = EarlyStopping(
    monitor=monitor_metric,
    patience=MODEL_CONFIG["early_stopping_patience"],
    min_delta=1e-4,
    restore_best_weights=True,
)
reduce_lr = ReduceLROnPlateau(
    monitor=monitor_metric,
    factor=MODEL_CONFIG["reduce_lr_factor"],
    patience=MODEL_CONFIG["reduce_lr_patience"],
    min_delta=1e-4,
    min_lr=MODEL_CONFIG["min_lr"],
    verbose=1,
)

mlp = create_model()

class_values = np.unique(output_data)
class_weights = compute_class_weight(class_weight="balanced", classes=class_values, y=output_data)
class_weights = np.sqrt(class_weights)
class_weights = class_weights / np.mean(class_weights)
class_weight_dict = dict(zip(class_values, class_weights))

fit_kwargs = dict(
    x=input_data,
    y=output_data,
    epochs=MODEL_CONFIG["epochs"],
    batch_size=MODEL_CONFIG["batch_size"],
    class_weight=class_weight_dict,
    callbacks=[early_stopping, reduce_lr],
)

if val_output is not None:
    fit_kwargs["validation_data"] = (val_input, val_output)

history = mlp.fit(**fit_kwargs)

# Plot training history

In [ ]:
epochs = np.arange(1, len(history.history['loss']) + 1)

pl.plot(epochs, history.history['loss'], label='Training Loss')
if 'val_loss' in history.history:
    pl.plot(epochs, history.history['val_loss'], label='Validation Loss')

pl.xlabel('Epochs')
pl.ylabel('Loss')
pl.legend()
pl.show()

# Performances

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns

class_names = ["Awake", "REM", "Non-REM"]

def plot_confusion_matrix(confusion_matrix, title):
    # Plot confusion matrix
    pl.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix.astype(int), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names)
    pl.title(title)
    pl.xlabel('Predicted')
    pl.ylabel('True')
    pl.show()

predictions = np.argmax(mlp.predict(val_input), axis=1)

predicted_counts = np.bincount(predictions, minlength=len(class_names))
print("Predicted samples per class on EEG_mouse_data_test.csv:")
for i, class_name in enumerate(class_names):
    print(f"{class_name}: {predicted_counts[i]}")

class_to_state = {0: "w", 1: "r", 2: "n"}
predicted_states = np.array([class_to_state[int(p)] for p in predictions])

test_feature_columns = list(mouse1.columns[1:11])
predicted_test_df = pd.DataFrame(test_data[:, :10], columns=test_feature_columns)
predicted_test_df.insert(0, "state", predicted_states)

output_csv_path = "EEG_mouse_data_test_with_predicted_state.csv"
predicted_test_df.to_csv(output_csv_path, index=False)
print(f"Exported predicted test dataset to {output_csv_path}")